In [ ]:
from google.colab import drive
drive.mount('/gdrive')

Mounted at /gdrive


In [ ]:
import tensorflow as tf
import numpy as np
import os
import random
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import shutil
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics import confusion_matrix
from PIL import Image
from distutils.dir_util import copy_tree
from sklearn.model_selection import StratifiedKFold
from keras.preprocessing.image import ImageDataGenerator

tfk = tf.keras
tfkl = tf.keras.layers
print(tf.__version__)

2.9.2


In [ ]:
%cd /gdrive/MyDrive/Colab Notebooks/Homework1

if(os.path.exists('training_data_final')):
  shutil.rmtree('training_data_final')
if(os.path.exists('training_kfold')):
  shutil.rmtree('training_kfold')
if(os.path.exists('validation_kfold')):
  shutil.rmtree('validation_kfold')



os.mkdir('validation_kfold')
os.mkdir('validation_kfold/Species1')
os.mkdir('validation_kfold/Species2')
os.mkdir('validation_kfold/Species3')
os.mkdir('validation_kfold/Species4')
os.mkdir('validation_kfold/Species5')
os.mkdir('validation_kfold/Species6')
os.mkdir('validation_kfold/Species7')
os.mkdir('validation_kfold/Species8')

/gdrive/MyDrive/Colab Notebooks/Homework1


In [ ]:
!rm -r training_kfold

rm: cannot remove 'training_kfold': No such file or directory


In [ ]:
!unzip training_dataset_homework1.zip

Archive:  training_dataset_homework1.zip
   creating: training_data_final/Species1/
  inflating: training_data_final/Species1/00000.jpg  
  inflating: training_data_final/Species1/00001.jpg  
  inflating: training_data_final/Species1/00002.jpg  
  inflating: training_data_final/Species1/00003.jpg  
  inflating: training_data_final/Species1/00004.jpg  
  inflating: training_data_final/Species1/00005.jpg  
  inflating: training_data_final/Species1/00006.jpg  
  inflating: training_data_final/Species1/00007.jpg  
  inflating: training_data_final/Species1/00008.jpg  
  inflating: training_data_final/Species1/00009.jpg  
  inflating: training_data_final/Species1/00010.jpg  
  inflating: training_data_final/Species1/00011.jpg  
  inflating: training_data_final/Species1/00012.jpg  
  inflating: training_data_final/Species1/00013.jpg  
  inflating: training_data_final/Species1/00014.jpg  
  inflating: training_data_final/Species1/00015.jpg  
  inflating: training_data_final/Species1/00016.jpg 

In [ ]:
training_data_final = 'training_data_final'


print("Duplico il training #1+2")
for subdir, dirs, files in os.walk(training_data_final):
    for class_dir in dirs:
        filenames = os.listdir(os.path.join(training_data_final, class_dir))
        for file in filenames:
            shutil.copy(os.path.join(training_data_final, class_dir, file), os.path.join(training_data_final, class_dir, "copy1_" + file))
            shutil.copy(os.path.join(training_data_final, class_dir, file), os.path.join(training_data_final, class_dir, "copy2_" + file))
            shutil.copy(os.path.join(training_data_final, class_dir, file), os.path.join(training_data_final, class_dir, "copy3_" + file))
            

Duplico il training #1+2


In [ ]:
'''if not os.path.exists('training_kfold'):
  os.mkdir('training_kfold')'''

!mv training_data_final training_kfold

In [ ]:
img_size = 96
input_shape = (img_size, img_size, 3)
batch_size = 256
epoch = 20

In [ ]:
seed = 42

random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
tf.random.set_seed(seed)
tf.compat.v1.set_random_seed(seed)

In [ ]:
import warnings
import logging

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)
tf.get_logger().setLevel('INFO')
tf.autograph.set_verbosity(0)

tf.get_logger().setLevel(logging.ERROR)
tf.get_logger().setLevel('ERROR')
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

In [ ]:
training_dir = 'training_kfold'

'''for subdir, dirs, files in os.walk(training_dir):
    for class_dir in dirs:
        filenames = os.listdir(os.path.join(training_dir, class_dir))
        for file in filenames:
            shutil.copy(os.path.join(training_dir, class_dir, file), os.path.join(training_dir, class_dir, "copy_" + file))'''

'for subdir, dirs, files in os.walk(training_dir):\n    for class_dir in dirs:\n        filenames = os.listdir(os.path.join(training_dir, class_dir))\n        for file in filenames:\n            shutil.copy(os.path.join(training_dir, class_dir, file), os.path.join(training_dir, class_dir, "copy_" + file))'

In [ ]:
root_path='/gdrive/MyDrive/Colab Notebooks/Homework1/'
datasetFolderName=root_path
MODEL_FILENAME=root_path+"model_cv.h5"
sourceFiles=[]
classLabels=['Species1', 'Species2', 'Species3','Species4','Species5','Species6','Species7','Species8']
X=[]
Y=[]

img_rows, img_cols =  96, 96 # input image dimensions
train_path=datasetFolderName+'training_kfold/'
validation_path=datasetFolderName+'validation_kfold/'

In [ ]:
def transferBetweenFolders(source, dest, splitRate): 
    global sourceFiles
    sourceFiles=os.listdir(source)
    if(len(sourceFiles)!=0):
        transferFileNumbers=int(len(sourceFiles)*splitRate)
        transferIndex=random.sample(range(0, len(sourceFiles)), transferFileNumbers)
        for eachIndex in transferIndex:
            shutil.move(source+str(sourceFiles[eachIndex]), dest+str(sourceFiles[eachIndex]))
    else:
        print("No file moved. Source empty!")
        
def transferAllClassBetweenFolders(source, dest, splitRate):
    for label in classLabels:
        transferBetweenFolders(datasetFolderName+source+'/'+label+'/', 
                               datasetFolderName+dest+'/'+label+'/', 
                               splitRate)

def my_metrics(y_true, y_pred):
    accuracy=accuracy_score(y_true, y_pred)
    print("Accuracy  : {}".format(accuracy))
    cm=confusion_matrix(y_true, y_pred)
    print(cm)
    return accuracy

In [ ]:
def prepareNameWithLabels(folderName):
    sourceFiles=os.listdir(datasetFolderName+'training_kfold/'+folderName)
    for val in sourceFiles:
        X.append(val)
        for i in range(len(classLabels)):
          if(folderName==classLabels[i]):
              Y.append(i)

In [ ]:
for i in range(len(classLabels)):
  prepareNameWithLabels(classLabels[i])

In [ ]:
X=np.asarray(X)
Y=np.asarray(Y)

In [ ]:
# Utility function to create folders and callbacks for training
from datetime import datetime
now = datetime.now().strftime('%b%d_%H-%M-%S')
def create_folders_and_callbacks(model_name):

  exps_dir = os.path.join('data_augmentation_experiments')
  if not os.path.exists(exps_dir):
      os.makedirs(exps_dir)

  exp_dir = os.path.join(exps_dir, model_name + '_' + str(now))
  if not os.path.exists(exp_dir):
      os.makedirs(exp_dir)
      
  callbacks = []

  # Model checkpoint
  # ----------------
  ckpt_dir = os.path.join(exp_dir, 'ckpts')
  if not os.path.exists(ckpt_dir):
      os.makedirs(ckpt_dir)

  ckpt_callback = tf.keras.callbacks.ModelCheckpoint(filepath=os.path.join(ckpt_dir, 'cp.ckpt'), 
                                                     save_weights_only=True, # True to save only weights
                                                     save_best_only=False) # True to save only the best epoch 
  callbacks.append(ckpt_callback)

  # Visualize Learning on Tensorboard
  # ---------------------------------
  tb_dir = os.path.join(exp_dir, 'tb_logs')
  if not os.path.exists(tb_dir):
      os.makedirs(tb_dir)
      
  # By default shows losses and metrics for both training and validation
  tb_callback = tf.keras.callbacks.TensorBoard(log_dir=tb_dir, 
                                               profile_batch=0,
                                               histogram_freq=1)  # if > 0 (epochs) shows weights histograms
  callbacks.append(tb_callback)

  # Early Stopping
  # --------------
  es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
  #callbacks.append(es_callback)

  return callbacks

In [ ]:
modello="KFOLD_DENSENET"

In [ ]:
#scarico la rete
supernet = tf.keras.applications.DenseNet201(
    include_top=False,
    weights="imagenet",
    input_tensor=None,
    pooling=None,
    classes=1000,
    classifier_activation="softmax",
    input_shape=(96,96,3)
)
def getModel():
  #aggiungo strati
  inputs = tfk.Input(shape=(img_size,img_size,3))
  x = tf.keras.layers.GaussianNoise(0.1)
  x = tfkl.Resizing(96, 96, interpolation="bicubic")(inputs)
  x = supernet(x)
  x = tfkl.GlobalAveragePooling2D(name='GlobalPooling')(x)
  x = tfkl.Flatten(name='Flattening')(x)
  x = tfkl.Dropout(0.3, seed=seed)(x)
  x = tfkl.Dense(
      256, 
      activation='relu',
      kernel_initializer = tfk.initializers.HeUniform(seed))(x)
  x = tfkl.Dropout(0.3, seed=seed)(x)
  outputs = tfkl.Dense(
      8, 
      activation='softmax',
      kernel_initializer = tfk.initializers.GlorotUniform(seed))(x)


  # Connect input and output through the Model class
  model = tfk.Model(inputs=inputs, outputs=outputs, name='model')

  # Compile the model
  model.compile(loss=tfk.losses.CategoricalCrossentropy(), optimizer=tfk.optimizers.Adam(), metrics='accuracy')

  ft_model = model
  version = 'densenet201'
  ft_percent = 0.02
  n_layers = len(ft_model.get_layer(version).layers) - 1

  # Set all DenseNet201 layers to True
  ft_model.get_layer(version).trainable = True
  

  # Freeze first N layers, e.g., until 14th
  for i, layer in enumerate(ft_model.get_layer(version).layers[:(int)(ft_percent*n_layers)]):
    layer.trainable=False
  
  ft_model.compile(loss=tfk.losses.CategoricalCrossentropy(), optimizer=tfk.optimizers.Adam(1e-4), metrics='accuracy')
  return ft_model

model=getModel()

In [ ]:
# ===============Stratified K-Fold======================
skf = StratifiedKFold(n_splits=5, shuffle=True)
skf.get_n_splits(X, Y)
foldNum=0

for train_index, val_index in skf.split(X, Y):
    #First cut all images from validation to train (if any exists)
    transferAllClassBetweenFolders('validation_kfold', 'training_kfold', 1.0)
    foldNum+=1
    print("Results for fold",foldNum)
    X_train, X_val = X[train_index], X[val_index]
    Y_train, Y_val = Y[train_index], Y[val_index]
    # Move validation images of this fold from train folder to the validation folder
    for eachIndex in range(len(X_val)):
        classLabel=''
        for i in range(len(classLabels)):
          if(Y_val[eachIndex]==i):
              classLabel=classLabels[i]
        #Then, copy the validation images to the validation folder
        shutil.move(datasetFolderName+'training_kfold/'+classLabel+'/'+X_val[eachIndex], 
                    datasetFolderName+'validation_kfold/'+classLabel+'/'+X_val[eachIndex])
        
    train_datagen = ImageDataGenerator(rotation_range=360,
                                        fill_mode='reflect',
                                        brightness_range=(0.2,1.8),
                                        channel_shift_range=150,
                                        shear_range=5,
                                        zoom_range=0.5,
                                        height_shift_range=50,
                                        width_shift_range=50,
                                        horizontal_flip=True,
                                        vertical_flip=True,
                                        preprocessing_function=tf.keras.applications.densenet.preprocess_input)
    validation_datagen = ImageDataGenerator(preprocessing_function=tf.keras.applications.densenet.preprocess_input)
        
    #Start ImageClassification Model
    train_generator = train_datagen.flow_from_directory(
        train_path,
        target_size=(img_rows, img_cols),
        batch_size=batch_size,
        class_mode='categorical'
        )
    validation_generator = validation_datagen.flow_from_directory(
        validation_path,
        target_size=(img_rows, img_cols),
        batch_size=batch_size,
        class_mode=None,  # only data, no labels
        shuffle=False)   
     # fit model
    history=model.fit_generator(train_generator, 
                        epochs=epoch)
    
    predictions = model.predict_generator(validation_generator, verbose=1)
    yPredictions = np.argmax(predictions, axis=1)
    true_classes = validation_generator.classes
    # evaluate validation performance
    print("***Performance on Validation data***")    
    valAcc = my_metrics(true_classes, yPredictions)

No file moved. Source empty!
No file moved. Source empty!
No file moved. Source empty!
No file moved. Source empty!
No file moved. Source empty!
No file moved. Source empty!
No file moved. Source empty!
No file moved. Source empty!
Results for fold 1
Found 11334 images belonging to 8 classes.
Found 2834 images belonging to 8 classes.
Epoch 1/20
45/45 [==============================] - 117s 2s/step - loss: 2.0083 - accuracy: 0.3043
Epoch 2/20
45/45 [==============================] - 68s 1s/step - loss: 1.3964 - accuracy: 0.5026
Epoch 3/20
45/45 [==============================] - 68s 2s/step - loss: 1.1572 - accuracy: 0.5866
Epoch 4/20
45/45 [==============================] - 71s 2s/step - loss: 1.0208 - accuracy: 0.6446
Epoch 5/20
45/45 [==============================] - 68s 2s/step - loss: 0.9311 - accuracy: 0.6721
Epoch 6/20
45/45 [==============================] - 69s 2s/step - loss: 0.8605 - accuracy: 0.6990
Epoch 7/20
45/45 [==============================] - 70s 2s/step - loss: 0.8

In [ ]:
# Save the best model
model.save(modello + "FT_" + str(now))
del model